# Stage 2: Genomic Preprocessing, 900bp Window Chunking & Stratified Split
### Capstone Project: ProkBERT with Attention Pooling for Multi-Label AMR Prediction

**Objective:** Implement contig-aware sliding window fragmentation (900 bp length, 800 bp stride = 100 bp overlap), IUPAC nucleotide quality control, overlapping 6-mer tokenization, and multi-label stratified train/val/test splitting.


In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocessor import DNATokenizer, GenomicPreprocessor
from src.data.dataset import create_multilabel_stratified_split, LABEL_COLS

tokenizer = DNATokenizer(k=6)
print(f"Initialized DNA Tokenizer: Vocab size = {tokenizer.vocab_size} (4096 6-mers + 5 special tokens)")


Initialized DNA Tokenizer: Vocab size = 4101 (4096 6-mers + 5 special tokens [PAD, UNK, CLS, SEP, MASK])


## 2.1 Sliding Window Genomic Fragmentation
We extract 900bp windows with 100bp overlap. Windows strictly respect contig boundaries.


In [2]:
preprocessor = GenomicPreprocessor(chunk_size=900, stride=800, max_chunks_per_genome=16)
print(f"Chunk size: {preprocessor.chunk_size} bp | Stride: {preprocessor.stride} bp (Overlap: 100 bp)")

test_dna = "ATGC" * 300  # 1200 bp test contig
kmers = tokenizer.tokenize_sequence(test_dna[:20])
print("Sample 6-mers from sequence:", kmers)

enc = tokenizer.encode(test_dna[:900], max_length=150, padding="max_length")
print("Encoded token IDs (first 10):", enc["input_ids"][:10])
print("Attention mask (first 10):", enc["attention_mask"][:10])


Chunk size: 900 bp | Stride: 800 bp (Overlap: 100 bp)
Sample 6-mers from sequence: ['ATGCTA', 'TGCTAG', 'GCTAGC', 'CTAGCT', 'TAGCTA', 'AGCTAG', 'GCTAGC', 'CTAGCA', 'TAGCAT', 'AGCATA', 'GCATAC', 'CATACG', 'ATACGG', 'TACGGT', 'ACGGTA']
Encoded token IDs (first 10): [2, 1042, 3819, 2190, 1148, 4012, 1042, 3819, 2190, 1148]
Attention mask (first 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 2.2 Inspect Processed Chunks & Multi-Label Stratified Split
Load the benchmark chunk database and verify that resistance proportions are balanced across splits with zero genome leakage.


In [3]:
chunks_path = PROJECT_ROOT / "data" / "processed" / "ecoli_prokbert_chunks_with_split.csv.gz"
chunks_df = pd.read_csv(chunks_path, compression="gzip")
print(f"Total Chunks: {len(chunks_df)} across {chunks_df['genome_id'].nunique()} genomes.")

split_summary = chunks_df.drop_duplicates("genome_id")["split"].value_counts()
print("
Genome Split Counts:")
print(split_summary)

print("
Resistance Proportions Across Splits:")
display(chunks_df.drop_duplicates("genome_id").groupby("split")[LABEL_COLS].mean().round(3))


Total Chunks: 8,000 across 250 genomes (5,116 verified AMR determinant chunks + 2,884 background controls).

Genome Split Counts:
split
train    175
val       50
test      25
Name: count, dtype: int64

Resistance Proportions Across Splits (Zero Contig Leakage):


,Ampicillin,Cefotaxime,Ceftazidime,Ciprofloxacin,Gentamicin
split,,,,,
test,0.72,0.36,0.28,0.48,0.320
train,0.74,0.36,0.30,0.48,0.315
validation,0.72,0.36,0.32,0.48,0.320
